# AI Agent MCP Demo

Minimal LangChain proof of concept that loads tools from the asyncroscopy MCP server and asks the model to use one or two of them. This will be useful to test our MCP server and new tools.

## Prereqs

Start the Tango stack and MCP server first:

```bash
uv run startup_scripts/run_servers.py
uv run startup_scripts/run_mcp.py --yaml configs/mcp.yaml
```

If the agent packages are missing from your environment, install them into the same project environment first:

```bash
uv sync --extra agent
# for local models do uv sync --extra agent --extra localagent
```

In [1]:
import os
from pprint import pprint

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient

In [2]:
MCP_URL = "http://127.0.0.1:8000/mcp"

client = MultiServerMCPClient(
    {
        "asyncroscopy": {
            "url": MCP_URL,
            "transport": "streamable_http",
        }
    }
)

tools = await client.get_tools()
print(f"Loaded {len(tools)} tool(s)")
print([tool.name for tool in tools[:10]])

Loaded 45 tool(s)
['get_data_from_key', 'list_devices', 'CAMERA_State', 'CAMERA_Status', 'DATA_State', 'DATA_Status', 'DATA_configure', 'DATA_get_config', 'DATA_register_path', 'DATA_register_save_path']


### Run cell below to use an OPENAI-COMPATIBLE model

In [ ]:
model = init_chat_model(
    model="YOUR MODEL NAME",
    model_provider="openai, google_genai, etc.", # Make sure to install the right provider package for the model you want to use
    api_key="YOUR API KEY", 
)

### Run cell below to use a LOCAL model

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

HF_MODEL_ID = r"C:\Users\Public\Desktop\Agents\Gemma4-31B"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
hf_model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

hf_pipeline = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.2,
    return_full_text=False,
)

model = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=hf_pipeline))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [7]:
agent = create_agent(model, tools)

def final_text(result):
    message = result["messages"][-1]
    return getattr(message, "content", message)

result = await agent.ainvoke({
    "messages": "List the available MCP devices or tools in one short sentence, then stop."
})
print(final_text(result))

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring


[{'type': 'text', 'text': 'The available devices are CAMERA, DATA, DigitalTwin, EDS, FLUCAM, SCAN, and STAGE.', 'extras': {'signature': 'EjQKMgERTTIPFv9j3y3+h2snAySjKTlHDyG4ep6/U2/5EbvTz7aulR1vyPiTwU6tEa0W+6Ok'}}]


In [8]:
tool_names = [tool.name.lower() for tool in tools]
imageish_tools = [
    tool_name
    for tool_name in tool_names
    if any(token in tool_name for token in ("image", "acquire", "scan", "camera", "detector"))
]

if imageish_tools:
    prompt = (
        "Use exactly one MCP tool to get a single image or acquisition result. "
        f"Prefer one of these tools if it fits: {', '.join(imageish_tools)}."
    )
else:
    prompt = "No obvious image tool was exposed, so just report that and stop."

result = await agent.ainvoke({"messages": prompt})
print(final_text(result))

Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key 'additionalProperties' is not suppor

[{'type': 'text', 'text': 'The spectrum acquisition was successful, and the resulting file is `spectrum_eds_20260706T094720097701.h5`.', 'extras': {'signature': 'EjQKMgERTTIPETsKZkEzHit65FofXy60qsYfEskNBUZkKZXOneBf44uVa02J2xQRlOJZ70Bc'}}]
